In [ ]:
import jax.numpy as jnp
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy.stats import spearmanr

from vizopt.templates.color import (
    oklab_to_rgb,
    optimize_colors,
    plot_colored_words,
    rgb_to_oklab,
    rgb_to_oklch,
)

In [ ]:
words = ["paper", "Papier", "papel", "palpiri", "carta", "karatasi"]
n_words = 9
words = [str(i_int) for i_int in range(n_words)]

In [ ]:
def edit_distance(a, b):
    """Levenshtein distance via DP."""
    a, b = a.lower(), b.lower()
    m, n = len(a), len(b)
    dp = list(range(n + 1))
    for i, ca in enumerate(a, 1):
        prev = dp[:]
        dp[0] = i
        for j, cb in enumerate(b, 1):
            dp[j] = prev[j-1] if ca == cb else 1 + min(prev[j], dp[j-1], prev[j-1])
    return dp[n]


In [ ]:
distances = pd.DataFrame(
    [[edit_distance(w1, w2) for w2 in words] for w1 in words],
    index=words,
    columns=words,
)
distances

In [ ]:
main_color = "#784086"

main_color = "#1CB61C"

In [ ]:
sparse_cb = lambda i, loss, *_: print(f"iter {i:4d}  loss={loss:.6f}") if i % 200 == 0 else None

colors, history = optimize_colors(
    distances,
    fixed_colors={words[0]: mcolors.to_rgb(main_color)},
    target_max_delta_e=0.1,
    learning_rate=0.05,
    target_L=0.75,
    n_iters=1000,
    callback=sparse_cb,
    coverage_weight=1.0,
    stress_weight=0.5
)
print(f"Final loss: {history[-1]['total']:.6f}")

In [ ]:
def plot_colors(rgb_list):
    _, ax = plt.subplots(figsize=(4, 2))
    for i_col, col in enumerate(rgb_list):
        ax.scatter(i_col, 0, color=col, s=200, edgecolors="0.3", linewidths=0.5)
    ax.set_yticks([])

    _, ax = plt.subplots(figsize=(3, 3))
    lab = np.array(rgb_to_oklab(jnp.array(rgb_list)))
    for r, g, b in zip(rgb_list, lab[:, 1], lab[:, 2]):
        ax.scatter(b, g, color=r, s=200, edgecolors="0.3", linewidths=0.5)
    ax.set_xlabel("b (OKLAB)")
    ax.set_ylabel("a (OKLAB)")
    ax.axhline(0, color="0.8", lw=0.5)
    ax.axvline(0, color="0.8", lw=0.5)

plot_colors(colors)

In [ ]:
pd.DataFrame(history).set_index("iteration").plot(marker=".")
plt.ylabel("Loss value")
plt.xlabel("Iteration")

In [ ]:
n = len(words)
pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
D = np.array(distances)

lab_opt = np.array(rgb_to_oklab(jnp.array(colors)))
color_dists_opt = [np.linalg.norm(lab_opt[i] - lab_opt[j]) for i, j in pairs]
rho, _ = spearmanr(color_dists_opt, [D[i, j] for i, j in pairs])
print(f"Spearman ρ = {rho:.3f}")

In [ ]:
n_restarts = 3
results = []
for seed in range(n_restarts):
    c, history = optimize_colors(
        distances,
        fixed_colors={words[0]: mcolors.to_rgb(main_color)},
        target_max_delta_e=0.2,
        learning_rate=0.05,
        n_iters=1000,
        seed=seed,
    )
    loss = history[-1]["total"]
    results.append((loss, c))
    print(f"seed={seed:2d}  loss={loss:.6f}")

best_loss, best_colors = min(results, key=lambda x: x[0])
print(f"\nBest loss = {best_loss:.6f}")

In [ ]:
plot_colors(best_colors)

In [ ]:
from IPython.display import HTML

from vizopt.animation import SnapshotCallback, animate
from vizopt.templates.color import build_color_input_parameters, build_color_problem

In [ ]:
input_parameters = build_color_input_parameters(
    distances,
    fixed_colors={words[0]: mcolors.to_rgb(main_color)},
    target_max_delta_e=0.25,
)
problem = build_color_problem(input_parameters)
snapshot_cb = SnapshotCallback(every=50)
optim_vars_opt, history = problem.optimize(n_iters=2000, track_every=200, callback=snapshot_cb)

In [ ]:
from vizopt.templates.color import _build_rgb

fig, ax = plt.subplots(figsize=(12, 3))

iterations = [it for it, _ in snapshot_cb.snapshots]
all_rgb = np.array([
    _build_rgb(optim_vars, problem.input_parameters)
    for _, optim_vars in snapshot_cb.snapshots
])  # shape: (n_snapshots, n_colors, 3)

n_snapshots, n_colors, _ = all_rgb.shape
xs = np.repeat(iterations, n_colors)
ys = np.tile(np.arange(n_colors), n_snapshots)
colors = all_rgb.reshape(-1, 3)

ax.scatter(xs, ys, color=colors, s=150, edgecolors="0.3", linewidths=0.5)

ax.set_xlabel("Iteration")
ax.set_ylabel("Color index")
labels = problem.input_parameters["labels"]
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))

all_lab = np.array(
    [rgb_to_oklab(rgb) for rgb in all_rgb]
)  # shape: (n_snapshots, n_colors, 3)

sizes = np.linspace(20, 200, n_snapshots)
all_a = all_lab[:, :, 1].reshape(-1)  # (n_snapshots * n_colors,)
all_b = all_lab[:, :, 2].reshape(-1)
all_s = np.repeat(sizes, n_colors)
all_c = all_rgb.reshape(-1, 3)

ax.scatter(
    all_b, all_a, color=all_c, s=all_s, edgecolors="0.3", linewidths=0.3, alpha=0.5
)

ax.set_xlabel("b (OKLAB)")
ax.set_ylabel("a (OKLAB)")
ax.axhline(0, color="0.8", lw=0.5)
ax.axvline(0, color="0.8", lw=0.5)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import SVG

from vizopt.animation import snapshots_to_animated_svg

svg = snapshots_to_animated_svg(problem, snapshot_cb.snapshots, fps=20)
SVG(data=svg)

In [ ]:
pd.DataFrame(history).set_index("iteration").plot(marker=".")
plt.ylabel("Loss value")
plt.xlabel("Iteration")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from vizopt.templates.color import rgb_to_oklab

colormaps = ['Pastel1', 'Pastel2', 'Paired', 'Accent', 'Dark2']

results = {}
for name in colormaps:
    cmap = plt.get_cmap(name)
    n = cmap.N
    rgb = np.array([cmap(i)[:3] for i in range(n)])
    lab = np.array(rgb_to_oklab(rgb))
    coverage = -(lab[:, 1].var() + lab[:, 2].var())
    results[name] = coverage
    print(f"{name:10s}  n={n:2d}  coverage={coverage:8.4f}  (a+b spread={-coverage:.4f})")

In [ ]:
fig, axes = plt.subplots(1, len(colormaps), figsize=(14, 4), sharex=True, sharey=True)

for ax, name in zip(axes, colormaps):
    cmap = plt.get_cmap(name)
    rgb = np.array([cmap(i)[:3] for i in range(cmap.N)])
    lab = np.array(rgb_to_oklab(rgb))
    for r, g, b in zip(rgb, lab[:, 1], lab[:, 2]):
        ax.scatter(b, g, color=r, s=200, edgecolors="0.3", linewidths=0.5)
    ax.set_title(name)
    ax.set_xlabel("b (OKLAB)")
    ax.axhline(0, color="0.8", lw=0.5)
    ax.axvline(0, color="0.8", lw=0.5)

axes[0].set_ylabel("a (OKLAB)")
plt.tight_layout()
plt.show()

In [ ]:
import networkx as nx

tree = nx.balanced_tree(r=2, h=2, create_using=nx.DiGraph)
tree.nodes

mapping = {}
queue = [(0, "")]
while queue:
    node, label = queue.pop(0)
    mapping[node] = label
    for i, child in enumerate(tree.successors(node)):
        queue.append((child, label + str(i)))

mapping[0] = "root"
tree = nx.relabel_nodes(tree, mapping)

nx.draw_networkx(tree)

In [ ]:
from vizopt.templates.layered_graph import (
    layered_graph_template,
    make_layered_graph_input_params,
)

layout_params = make_layered_graph_input_params(tree)
problem = layered_graph_template.instantiate(layout_params)

optim_vars, history = problem.optimize(n_iters=10000, learning_rate=2*1e-2)

problem.plot_configuration(optim_vars, layout_params)

In [ ]:
_, ax = plt.subplots(figsize=(4, 4))
node_xys = optim_vars["node_xys"]
for i, j in problem.input_parameters['edge_indices']:
        ax.annotate(
            "",
            xy=node_xys[j],
            xytext=node_xys[i],
            arrowprops=dict(arrowstyle="->", color="gray", lw=1.5),
        )

ax.scatter(node_xys[:, 0], node_xys[:, 1], s=80, zorder=3)

In [ ]:
nodes = list(tree.nodes)
undirected = tree.to_undirected()
distances = pd.DataFrame(
    dict(nx.all_pairs_shortest_path_length(undirected)),
    index=nodes,
    columns=nodes,
).loc[nodes, nodes]
distances

In [ ]:
from vizopt.templates.layered_graph import (
    layered_graph_template,
    make_layered_graph_input_params,
)

# Optimize colors based on tree distances
colors, _ = optimize_colors(
    distances,
    target_max_delta_e=0.2,
    learning_rate=0.05,
    n_iters=2000,
    coverage_weight=0.2,
)
color_map = dict(zip(nodes, colors))

# Optimize layout positions (top-down tree: preferred edge vector points downward)
# layout_params = make_layered_graph_input_params(tree, min_distance=1.0, preferred_edge_vector=(0.0, -1.0))
# layout_problem = layered_graph_template.instantiate(layout_params)
# optim_vars, _ = layout_problem.optimize(n_iters=2000, learning_rate=0.01)

node_xys = optim_vars["node_xys"]
node_names = layout_params["node_names"]
pos = {
    name: (float(node_xys[i, 0]), float(node_xys[i, 1]))
    for i, name in enumerate(node_names)
}

# Plot
fig, ax = plt.subplots(figsize=(5, 4))
nx.draw_networkx(
    tree,
    pos=pos,
    node_color=[color_map[n] for n in tree.nodes],
    with_labels=True,
    ax=ax,
    node_size=700,
    font_size=9,
    arrows=True,
)
ax.set_title("Tree with optimized positions and colors")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()